# Lab 14: Ablation et Raffinement Ciblé (MLE-STAR Component)

**Navigation** : [Lab 13 <<](Lab13-Web-Search-SOTA.ipynb) | [Index](../../README.md) | [>> Lab 15](Lab15-Kaggle-Challenge.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Identifier les blocs de code dans un pipeline ML
2. Réaliser des études d'ablation pour prioriser les améliorations
3. Raffiner de manière ciblée les composants critiques
4. Optimiser l'effort d'amélioration d'un modèle

### Prérequis
- Lab 13 (Web Search SOTA) complété
- Connaissance des pipelines ML
- Configuration multi-provider active

### Durée estimée : 40-50 minutes

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from enum import Enum

from config import get_settings
from utils import LLMClient

print("Imports OK : json, re, numpy, dataclasses, Enum, config, utils")

Imports OK : json, re, numpy, dataclasses, Enum, config, utils


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. Code Block Identification

In [1]:
@dataclass
class CodeBlock:
    id: str
    name: str
    description: str
    code: str
    importance: float = 0.0

class BlockType(Enum):
    FEATURE_ENGINEERING = "feature_engineering"
    MODEL_SELECTION = "model_selection"
    HYPERPARAMETER_TUNING = "hyperparameter_tuning"
    TRAINING = "training"
    EVALUATION = "evaluation"
    ENSEMBLE = "ensemble"

print("Dataclasses definies : CodeBlock (id, nom, description, code, importance), BlockType (6 types de blocs ML)")

Dataclasses definies : CodeBlock (id, nom, description, code, importance), BlockType (6 types de blocs ML)


Analyseur de blocs de code pour identifier les composants a tester.

In [1]:
class CodeBlockAnalyzer:
    """Identifie les blocs de code dans un pipeline ML."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def identify_blocks(self, code: str) -> List[CodeBlock]:
        """Identifie les blocs logiques dans le code."""
        prompt = f"""Analyse ce code ML et identifie les blocs logiques.

CODE:
```python
{code[:2000]}
```

Identifie 3-5 blocs (feature engineering, model, training, etc.).
Format JSON:
[
  {{"id": "block1", "name": "...", "description": "...", "code": "..."}}]
]

JSON:"""

        response = self.llm.generate(prompt, temperature=0.2)

        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return [CodeBlock(**b) for b in data]
        except:
            pass
        return []

print("Classe CodeBlockAnalyzer definie : identification de blocs logiques dans un pipeline ML via LLM")

Classe CodeBlockAnalyzer definie : identification de blocs logiques dans un pipeline ML via LLM


## 3. Ablation Study Simulator

Une **etude d'ablation** (ablation study) consiste a retirer systematiquement des composants
d'un systeme (bloc de code, couche de reseau, feature, module) pour mesurer leur contribution
relative aux performances globales. Originaire des neurosciences ou elle designait la lesion
controlee de regions cerebrales, la methode est devenue un standard d'evaluation en machine
learning : elle permet de distinguer les composants essentiels de ceux qui n'apportent que
des gains marginaux, et ainsi de cibler les efforts d'optimisation sur les zones a fort impact.

> Reference methodologique : Meyes, R., Schuman, C., Sakhavi, S. et al. (2019).
> *Ablation Studies in Artificial Neural Networks*. arXiv:1901.08644.
> https://arxiv.org/abs/1901.08644

Dans ce lab, le simulateur demande a un LLM d'estimer (sans execution reelle) l'importance
de chaque bloc identifie afin de guider le raffinement cible - une composante de la
strategie MLE-STAR (*Machine Learning Engineering Agent via Search and Targeted Refinement*,
Guo et al. 2025, arXiv:2506.15692) qui decompose le pipeline en blocs testables.

In [1]:
class AblationStudy:
    """Simule des etudes d'ablation pour prioriser les ameliorations."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def estimate_importance(self, blocks: List[CodeBlock], task: str) -> List[CodeBlock]:
        """Estime l'importance relative de chaque bloc."""
        blocks_desc = "\n".join([
            f"- {b.id}: {b.name} - {b.description[:50]}"
            for b in blocks
        ])

        prompt = f"""Estime l'importance de chaque bloc pour la tache: {task}

BLOCS:
{blocks_desc}

Donne un score d'importance de 0.0 a 1.0 pour chaque bloc.
Format JSON:
[
  {{"id": "block1", "importance": 0.9}},
  {{"id": "block2", "importance": 0.6}}
]

JSON:"""

        response = self.llm.generate(prompt, temperature=0.2)

        try:
            match = re.search(r'\[.*\]', response, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                importance_map = {d['id']: d['importance'] for d in data}
                for block in blocks:
                    block.importance = importance_map.get(block.id, 0.5)
        except:
            # Default importance
            for i, block in enumerate(blocks):
                block.importance = 0.8 - i * 0.1

        return sorted(blocks, key=lambda b: b.importance, reverse=True)

print("Classe AblationStudy definie : estimation de l'importance relative des blocs (0.0-1.0) via LLM")

Classe AblationStudy definie : estimation de l'importance relative des blocs (0.0-1.0) via LLM


## 4. Targeted Refinement

In [1]:
class TargetedRefiner:
    """Raffine de maniere ciblee les composants critiques."""

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def refine_block(self, block: CodeBlock, task: str, current_score: float) -> str:
        """Genere une version amelioree d'un bloc."""
        prompt = f"""Ameliore ce bloc de code ML pour la tache: {task}

BLOC: {block.name}
DESCRIPTION: {block.description}
CODE ACTUEL:
```python
{block.code[:1000]}
```

SCORE ACTUEL: {current_score:.3f}

Genere une version amelioree qui pourrait augmenter le score.
Focus sur les optimisations simples et efficaces.

```python
# Code ameliore
```"""

        response = self.llm.generate(prompt, temperature=0.3)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else block.code

print("Classe TargetedRefiner definie : raffinement cible de blocs critiques via LLM")

Classe TargetedRefiner definie : raffinement cible de blocs critiques via LLM


## 5. MLE-STAR Ablation Pipeline

In [1]:
class MLEStarAblation:
    """Pipeline d'ablation complet de MLE-STAR."""

    def __init__(self):
        self.llm = LLMClient()
        self.block_analyzer = CodeBlockAnalyzer(self.llm)
        self.ablation_study = AblationStudy(self.llm)
        self.refiner = TargetedRefiner(self.llm)

    def analyze_and_refine(self, code: str, task: str, current_score: float = 0.5) -> Dict:
        """Analyse le code, identifie les blocs, et suggere des raffinements."""
        print("[ANALYZER] Identification des blocs...")
        blocks = self.block_analyzer.identify_blocks(code)
        print(f"  - {len(blocks)} blocs identifies")

        if not blocks:
            return {'success': False, 'error': 'Impossible d identifier les blocs'}

        print("\n[ABLATION] Estimation de l'importance...")
        blocks = self.ablation_study.estimate_importance(blocks, task)

        print("  Resultats:")
        for b in blocks[:3]:
            print(f"    - {b.name}: {b.importance:.2f}")

        # Raffiner le bloc le plus important
        top_block = blocks[0]
        print(f"\n[REFINER] Raffinement de '{top_block.name}'...")
        refined_code = self.refiner.refine_block(top_block, task, current_score)

        return {
            'success': True,
            'blocks': [(b.name, b.importance) for b in blocks],
            'top_block': top_block.name,
            'refined_code': refined_code
        }

print("Classe MLEStarAblation definie : pipeline complet Analyze -> Ablation -> Refine")

[ANALYZER] Identification des blocs...


## 6. Test avec un Exemple de Code

In [8]:
# Exemple de code ML a analyser
sample_code = '''
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Feature Engineering
df = pd.read_csv('data.csv')
df['feature_ratio'] = df['feature_a'] / (df['feature_b'] + 1e-6)
df['feature_log'] = np.log1p(df['feature_c'])

X = df.drop('target', axis=1)
y = df['target']

# Model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Training & Evaluation
scores = cross_val_score(model, X, y, cv=5)
print(f"Accuracy: {scores.mean():.4f}")
'''

print("Code a analyser:")
print(sample_code[:300] + "...")

Code a analyser:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Feature Engineering
df = pd.read_csv('data.csv')
df['feature_ratio'] = df['feature_a'] / (df['feature_b'] + 1e-6)
df['feature_log'] = np.log1p(df['feature_c'])

X = df.drop...


Test du pipeline d'ablation sur un exemple de code ML.

In [9]:
# Test du pipeline d'ablation
pipeline = MLEStarAblation()

result = pipeline.analyze_and_refine(
    code=sample_code,
    task="binary classification tabular",
    current_score=0.75
)

print("\n" + "="*50)
print("RESULTAT DE L'ABLATION:")
print("="*50)
print(f"\nBlocs identifies:")
for name, imp in result['blocks']:
    print(f"  - {name}: importance {imp:.2f}")

print(f"\nBloc prioritaire: {result['top_block']}")

[ANALYZER] Identification des blocs...


  - 4 blocs identifies
Provider List: https://docs.litellm.ai/docs/providers



[ABLATION] Estimation de l'importance...


  Resultats:
Provider List: https://docs.litellm.ai/docs/providers


    - Data Loading & Feature Engineering: 1.00
    - Training & Evaluation: 0.90
    - Model Initialization: 0.80

[REFINER] Raffinement de 'Data Loading & Feature Engineering'...



Provider List: https://docs.litellm.ai/docs/providers


RESULTAT DE L'ABLATION:

Blocs identifies:
  - Data Loading & Feature Engineering: importance 1.00
  - Training & Evaluation: importance 0.90
  - Model Initialization: importance 0.80
  - Imports: importance 0.30

Bloc prioritaire: Data Loading & Feature Engineering


## 7. Affichage du Code Raffine

In [10]:
if result['success'] and result.get('refined_code'):
    print("\n" + "="*50)
    print("CODE RAFFINE:")
    print("="*50)
    print(result['refined_code'][:600] + "..." if len(result['refined_code']) > 600 else result['refined_code'])


CODE RAFFINE:
import pandas as pd
import numpy as np

# Chargement des données
df = pd.read_csv('data.csv')

# Séparation immédiate de X et y pour éviter toute fuite de données (data leakage)
X = df.drop('target', axis=1)
y = df['target']

# 1. Gestion des valeurs manquantes (Crucial pour les données tabulaires)
# Remplissage des valeurs numériques par la médiane (plus robuste aux valeurs extrêmes que la moyenne)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# 2. Feature Engineering enrichi et robuste
# Ratio (existant)
X['feature_ra...


## 8. Resume du Lab
### Ce que nous avons implemente
1. **CodeBlockAnalyzer**: Identifie les blocs logiques
2. **AblationStudy**: Estime l'importance relative
3. **TargetedRefiner**: Ameliore les composants critiques
### Principe MLE-STAR
L'ablation permet de:
- Ne pas gaspiller d'efforts sur des composants peu impactants
- Focus sur les zones a haut potentiel d'amelioration
- Iterer rapidement sur les changements importants
### Limitations
- L'importance est estimee par le LLM (pas testee reellement)
- Une vraie ablation necessite d'executer le code
- Le raffinement est suggestif, pas garanti
### Prochaine etape
- **Lab 15**: Kaggle Challenge avec MLE-STAR complet

## Exercice : Ablation sur votre propre pipeline

Appliquez l'etude d'ablation a un pipeline ML de votre choix pour identifier les composants a optimiser.

### Objectifs
1. Ecrire un pipeline ML simple (3-5 blocs)
2. Analyser l'importance de chaque bloc
3. Raffiner le bloc le plus critique
4. Comparer les resultats avant/apres

### Instructions



In [11]:
# TODO: Definissez votre propre pipeline ML
mon_pipeline = '''
# Exemple: classification, regression, ou clustering
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

# 1. Chargement des donnees
# 2. Preprocessing
# 3. Feature engineering
# 4. Entrainement modele
# 5. Evaluation
'''

# TODO: Analysez les blocs
pipeline = MLEStarAblation()
result = pipeline.analyze_and_refine(
    code=mon_pipeline,
    task="votre tache ici",  # Ex: "multiclass classification"
    current_score=0.70  # Score actuel hypothetique
)

# TODO: Affichez les blocs identifies
for name, importance in result['blocks']:
    print(f"{name}: {importance:.2f}")

# TODO: Testez manuellement le code raffine
# Comparez avec une execution reelle si possible

[ANALYZER] Identification des blocs...


  - 3 blocs identifies
Provider List: https://docs.litellm.ai/docs/providers



[ABLATION] Estimation de l'importance...


  Resultats:
Provider List: https://docs.litellm.ai/docs/providers


    - Importation des dépendances: 0.90
    - Entraînement et Évaluation: 0.90
    - Préparation des données: 0.80

[REFINER] Raffinement de 'Importation des dépendances'...


Importation des dépendances: 0.90
Provider List: https://docs.litellm.ai/docs/providers


Entraînement et Évaluation: 0.90
Préparation des données: 0.80


## Exercice : Impact du Feature Engineering sur l'Importance

Creez un pipeline ML avec des blocs de feature engineering specifiques et analysez comment l'ablation de chaque bloc de features affecte l'importance estimee par le LLM.

### Objectifs
1. Definir 3 blocs de features (original, ratio, polynomial)
2. Analyser l'importance estimee de chaque bloc
3. Rédiger un rapport comparatif des resultats d'ablation

**Indice :**
- Bloc 1: features originales (`age`, `tenure`, `monthly_charges`)
- Bloc 2: features ratio (`charges_per_month = total_charges / tenure`)
- Bloc 3: features polynomiales (`age_squared`, `tenure_log`)
- Observez quel bloc le LLM estime le plus important et pourquoi

In [12]:
# Exercice : Ablation guidee sur le feature engineering
# Objectif : Comparer l'impact de differents blocs de features

# TODO: Definissez 3 pipelines avec des blocs de features differents
pipeline_original = """
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Features originales uniquement
df = pd.read_csv('churn.csv')
X = df[['age', 'tenure', 'monthly_charges', 'total_charges']]
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print(f"Accuracy: {score:.4f}")
"""

pipeline_avec_ratios = """
# TODO etudiant : ajoutez des features ratio
# Exemple: df['charges_per_month'] = df['total_charges'] / (df['tenure'] + 1)
# Exemple: df['charges_age_ratio'] = df['monthly_charges'] / df['age']
"""

pipeline_avec_polynomial = """
# TODO etudiant : ajoutez des features polynomiales
# Exemple: df['age_squared'] = df['age'] ** 2
# Exemple: df['tenure_log'] = np.log1p(df['tenure'])
"""

# TODO: Analysez chaque pipeline avec le pipeline d'ablation
# ablation = MLEStarAblation()
# for name, code in [("original", pipeline_original),
#                     ("ratios", pipeline_avec_ratios),
#                     ("polynomial", pipeline_avec_polynomial)]:
#     result = ablation.analyze_and_refine(code, "customer churn prediction", current_score=0.75)
#     print(f"\n--- {name} ---")
#     for b_name, importance in result['blocks']:
#         print(f"  {b_name}: {importance:.2f}")

print("Exercice a completer : ablation guidee sur le feature engineering")

Exercice a completer


## Exercice : Strategie d'Ensemble par Ablation

Utilisez l'ablation pour determiner la meilleure strategie d'ensemble (voting, stacking, blending) pour un pipeline ML. L'objectif est de comparer les approches d'ensemble et de choisir la plus adaptee.

### Objectifs
1. Definir 3 strategies d'ensemble dans un pipeline ML
2. Appliquer l'ablation pour comparer leur importance relative
3. Recommander la meilleure strategie avec justification

**Indice :**
- Voting : `VotingClassifier` avec plusieurs modeles
- Stacking : `StackingClassifier` avec meta-learner
- Blending : predictions sur un validation set
- L'ablation permet de determiner si l'ensemble apporte reellement un gain vs. un seul modele

In [13]:
# Exercice : Comparaison de strategies d'ensemble par ablation
# Objectif : Determiner la meilleure strategie d'ensemble

# TODO: Definissez un pipeline avec strategie d'ensemble
ensemble_pipeline = """
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Chargement et preparation
df = pd.read_csv('data.csv')
X = df.drop('target', axis=1)
y = df['target']

# Modeles individuels
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000)

# Strategie d'ensemble: Voting
ensemble = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr)],
    voting='soft'
)

# Evaluation
scores = cross_val_score(ensemble, X, y, cv=5, scoring='roc_auc')
print(f"Ensemble AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")
"""

# TODO: Analysez les blocs avec le pipeline d'ablation
# pipeline_ablation = MLEStarAblation()
# result = pipeline_ablation.analyze_and_refine(
#     code=ensemble_pipeline,
#     task="binary classification with ensemble",
#     current_score=0.80
# )
# 
# print("Blocs identifies:")
# for name, importance in result['blocks']:
#     print(f"  {name}: {importance:.2f}")
# print(f"\nBloc prioritaire: {result['top_block']}")

# TODO: Comparez avec un pipeline sans ensemble (modele unique)
# Quel est le gain estime de l'ensemble par rapport au modele seul ?
gain_estime = None  # TODO etudiant : estimez le gain (ex: +0.02 AUC)

# TODO: Recommandez la meilleure strategie
# Justification : Voting est plus simple, Stacking plus performant mais plus couteux
recommandation = None  # TODO etudiant : "voting" / "stacking" / "single_model"

print("Exercice a completer : comparaison de strategies d'ensemble par ablation")

Exercice a completer


### Questions de reflexion
- L'estimation d'importance par le LLM correspond-elle a votre intuition ?
- Comment pourriez-vous valider l'importance experimentalement ?
- Quelles seraient les limites d'une ablation automatisee ?


## References

- Meyes, R., Schuman, C., Sakhavi, S., et al. (2019). *Ablation Studies in Artificial Neural Networks*. arXiv:1901.08644. https://arxiv.org/abs/1901.08644
- Guo, Q., et al. (2025). *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted Refinement*. Google Research. arXiv:2506.15692. https://arxiv.org/abs/2506.15692
- Xi, Z., et al. (2025). *The Rise and Potential of Large Language Model Based Agents: A Survey*. arXiv:2309.07864.